In [2]:
"""
Final Project – 3D Voxel Classification Pipeline (Updated)

"""

# -------------------- Imports --------------------
import numpy as np
import pandas as pd
import laspy
import open3d as o3d
from sklearn.decomposition import PCA
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import os

In [3]:



# =====================================================
# 1️⃣ LOAD POINT CLOUD AND REMOVE OUTLIERS
# =====================================================
print("\n=== STEP 1: Load and clean point cloud ===")

input_file = "final_project_segment.laz"
las = laspy.read(input_file)
points = np.vstack((las.x, las.y, las.z)).T
print(f"Loaded {len(points):,} points from {input_file}")

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd, ind = pcd.remove_statistical_outlier(nb_neighbors=50, std_ratio=5.0)
points_filtered = np.asarray(pcd.points)
print(f"Removed {len(points) - len(points_filtered):,} outliers → {len(points_filtered):,} remaining")

np.save("filtered_points_final.npy", points_filtered)


=== STEP 1: Load and clean point cloud ===
Loaded 1,988,927 points from final_project_segment.laz
Removed 10,550 outliers → 1,978,377 remaining


In [4]:


# =====================================================
# 2️⃣ VOXELIZATION AND VOXEL IDS
# =====================================================
print("\n=== STEP 2: Voxelization ===")

voxel_size = 2.0  # meter
min_coords = np.floor(points_filtered.min(axis=0))
max_coords = np.ceil(points_filtered.max(axis=0))
dims = np.ceil((max_coords - min_coords) / voxel_size).astype(int)

coords = np.floor((points_filtered - min_coords) / voxel_size).astype(int)
coords = np.clip(coords, 0, dims - 1)
voxel_ids = np.ravel_multi_index((coords[:, 0], coords[:, 1], coords[:, 2]), dims=dims)

unique_ids, counts = np.unique(voxel_ids, return_counts=True)
valid_ids = unique_ids[counts >= 10]
mask_valid = np.isin(voxel_ids, valid_ids)
filtered_points_final = points_filtered[mask_valid]
voxel_ids_filtered = voxel_ids[mask_valid]

print(f"Kept {len(filtered_points_final):,} points in {len(valid_ids):,} voxels")

np.save("voxel_ids_filtered.npy", voxel_ids_filtered)
np.save("filtered_points_final.npy", filtered_points_final)


=== STEP 2: Voxelization ===
Kept 1,950,756 points in 26,617 voxels


In [5]:
# =====================================================
# 3️⃣ PCA FEATURE EXTRACTION
# =====================================================
print("\n=== STEP 3: Feature extraction ===")

voxel_features = []
voxel_normals = {}

for voxel_id in valid_ids:
    pts = filtered_points_final[voxel_ids_filtered == voxel_id]
    if len(pts) < 10:
        continue

    pca = PCA(n_components=3).fit(pts)
    λ1, λ2, λ3 = np.sort(pca.explained_variance_)[::-1]
    if λ1 == 0:
        continue

    n = pca.components_[-1]
    if n[2] < 0:
        n = -n
    voxel_normals[voxel_id] = n

    voxel_features.append({
        "voxel_id": voxel_id,
        "n_points": len(pts),
        "linearity": (λ1 - λ2) / λ1,
        "planarity": (λ2 - λ3) / λ1,
        "scattering": λ3 / λ1,
        "omnivariance": (λ1 * λ2 * λ3) ** (1/3),
        "sum_ev": λ1 + λ2 + λ3,
        "anisotropy": (λ1 - λ3) / λ1,
        "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
        "change_curvature": λ3 / (λ1 + λ2 + λ3),
        "z_range": pts[:, 2].max() - pts[:, 2].min(),
        "plane_std": np.std(pca.transform(pts)[:, 2])
    })

df_features = pd.DataFrame(voxel_features)
print(f"Extracted {len(df_features)} voxels with PCA features")



=== STEP 3: Feature extraction ===


C:\Users\aasne\AppData\Local\Temp\ipykernel_16824\331285878.py:33: RuntimeWarning: divide by zero encountered in log
  "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
C:\Users\aasne\AppData\Local\Temp\ipykernel_16824\331285878.py:33: RuntimeWarning: invalid value encountered in scalar multiply
  "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
C:\Users\aasne\AppData\Local\Temp\ipykernel_16824\331285878.py:33: RuntimeWarning: divide by zero encountered in log
  "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
C:\Users\aasne\AppData\Local\Temp\ipykernel_16824\331285878.py:33: RuntimeWarning: invalid value encountered in scalar multiply
  "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
C:\Users\aasne\AppData\Local\Temp\ipykernel_16824\331285878.py:33: RuntimeWarning: divide by zero encountered in log
  "eigentropy": -(λ1*np.log(λ1) + λ2*np.log(λ2) + λ3*np.log(λ3)),
C:\Users\aasne\AppData\Local\Temp\ipykernel_16824\331285878.py:

Extracted 26617 voxels with PCA features


In [6]:
# =====================================================
# 4️⃣ VOXEL CENTERS
# =====================================================
print("\n=== STEP 4: Compute voxel centers ===")
voxel_centers = np.array([
    filtered_points_final[voxel_ids_filtered == vid].mean(axis=0)
    for vid in df_features["voxel_id"].values
])
np.save("voxel_centers.npy", voxel_centers)
print(f"Saved {len(voxel_centers)} voxel centers → voxel_centers.npy")


=== STEP 4: Compute voxel centers ===
Saved 26617 voxel centers → voxel_centers.npy


In [12]:
# =====================================================
# 5️⃣ ADDITIONAL FEATURES: 2D z-range, 3D/2D density ratio, geometric curvature
# =====================================================
print("\n=== STEP 5: Compute additional geometric features (2D z-range, density ratio, curvature) ===")

# KD-trees for 2D and 3D neighborhoods
points_tree_2d = cKDTree(filtered_points_final[:, :2])
points_tree_3d = cKDTree(filtered_points_final)
voxel_tree = cKDTree(voxel_centers)

R2D = voxel_size * 1.0   # radius for 2D cylindrical neighborhood
R3D = voxel_size * 1.0   # radius for 3D spherical neighborhood
Rcurv = voxel_size * 2.0 # radius for curvature (search neighboring voxels)

zrange2d_list = []
density_ratio_list = []
geom_curv_list = []

for i, c in enumerate(voxel_centers):
    # 2D neighborhood (XY plane)
    idx2d = points_tree_2d.query_ball_point(c[:2], r=R2D)
    if len(idx2d) > 0:
        z_vals = filtered_points_final[idx2d, 2]
        zrange2d = z_vals.max() - z_vals.min()
    else:
        zrange2d = 0.0

    # 3D neighborhood (sphere)
    idx3d = points_tree_3d.query_ball_point(c, r=R3D)
    n2d = len(idx2d)
    n3d = len(idx3d)
    density_ratio = n3d / (n2d + 1e-6)

    # Geometric curvature: difference of normals with neighboring voxels
    curv_neighbors = voxel_tree.query_ball_point(c, r=Rcurv)
    if len(curv_neighbors) > 1:
        n_i = voxel_normals[df_features["voxel_id"].iloc[i]]
        diffs = []
        for j in curv_neighbors:
            n_j = voxel_normals[df_features["voxel_id"].iloc[j]]
            diffs.append(np.linalg.norm(n_i - n_j))
        geom_curv = np.mean(diffs)
    else:
        geom_curv = 0.0

    # Store features
    zrange2d_list.append(zrange2d)
    density_ratio_list.append(density_ratio)
    geom_curv_list.append(geom_curv)

# Add new features to DataFrame
df_features["z_range_2d"] = zrange2d_list
df_features["density_ratio_3d_2d"] = density_ratio_list
df_features["geom_curvature"] = geom_curv_list

print(f"Added new features: z_range_2d, density_ratio_3d_2d, geom_curvature for {len(df_features)} voxels")



=== STEP 5: Compute additional geometric features (2D z-range, density ratio, curvature) ===
Added new features: z_range_2d, density_ratio_3d_2d, geom_curvature for 26617 voxels


In [13]:
# =====================================================
# 6️⃣ MANUAL LABELING (FROM CLOUDCOMPARE)
# =====================================================
print("\n=== STEP 6: Load manual labels ===")

def read_las_coords(filepath):
    las = laspy.read(filepath)
    return np.vstack((las.x, las.y, las.z)).T

buildings = read_las_coords("voxel_centers_building.las")
trees = read_las_coords("voxel_centers_trees.las")
terrain = read_las_coords("voxel_centers_terrain.las")

tree = cKDTree(voxel_centers)
building_ids = df_features["voxel_id"].values[tree.query(buildings, k=1)[1]]
tree_ids = df_features["voxel_id"].values[tree.query(trees, k=1)[1]]
terrain_ids = df_features["voxel_id"].values[tree.query(terrain, k=1)[1]]

df_features["label"] = "unlabeled"
df_features.loc[df_features["voxel_id"].isin(building_ids), "label"] = "building"
df_features.loc[df_features["voxel_id"].isin(tree_ids), "label"] = "tree"
df_features.loc[df_features["voxel_id"].isin(terrain_ids), "label"] = "terrain"

print(df_features["label"].value_counts())



=== STEP 6: Load manual labels ===
label
unlabeled    24742
tree          1274
building       399
terrain        202
Name: count, dtype: int64


In [15]:

# =====================================================
# 7️⃣ TRAIN CATBOOST MODEL
# =====================================================
print("\n=== STEP 7: Train CatBoost model ===")

df = df_features.copy()

# Log-transformed features
df["scattering_log2"] = np.log2(df["scattering"] + 1e-6)
df["plane_std_log2"] = np.log2(df["plane_std"] + 1e-6)
df["z_range_log2"] = np.log2(df["z_range"] + 1e-6)
df["density_ratio_log2"] = np.log2(df["density_ratio_3d_2d"] + 1e-6)
df["geom_curvature_log2"] = np.log2(df["geom_curvature"] + 1e-6)

feature_cols = [
    "linearity", "planarity", "scattering_log2", "omnivariance", "sum_ev",
    "anisotropy", "eigentropy", "change_curvature",
    "z_range", "plane_std_log2",
    "z_range_2d", "density_ratio_3d_2d", "geom_curvature"
]


train_df = df[df["label"] != "unlabeled"]
X, y = train_df[feature_cols], train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=300, learning_rate=0.1, depth=6,
    loss_function='MultiClass', verbose=100
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nClassification report:")
print(classification_report(y_test, y_pred))


=== STEP 7: Train CatBoost model ===
0:	learn: 0.9376637	total: 4.75ms	remaining: 1.42s
100:	learn: 0.0161059	total: 477ms	remaining: 939ms
200:	learn: 0.0077361	total: 913ms	remaining: 450ms
299:	learn: 0.0047547	total: 1.35s	remaining: 0us

Classification report:
              precision    recall  f1-score   support

    building       0.98      1.00      0.99        80
     terrain       1.00      0.97      0.99        40
        tree       1.00      1.00      1.00       255

    accuracy                           0.99       375
   macro avg       0.99      0.99      0.99       375
weighted avg       0.99      0.99      0.99       375



In [16]:
# =====================================================
# 8️⃣ CLASSIFY ALL VOXELS AND POINTS
# =====================================================
print("\n=== STEP 8: Predict all voxels and points ===")

df["pred_label"] = model.predict(df[feature_cols]).flatten()
print(df["pred_label"].value_counts())

color_map = {
    "building": [1, 0, 0],
    "tree": [0, 1, 0],
    "terrain": [0.5, 0.25, 0],
    "unlabeled": [0.7, 0.7, 0.7]
}

colors_voxel = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in df["pred_label"].values])
pcd_vox = o3d.geometry.PointCloud()
pcd_vox.points = o3d.utility.Vector3dVector(voxel_centers)
pcd_vox.colors = o3d.utility.Vector3dVector(colors_voxel)

vis = o3d.visualization.Visualizer()
vis.create_window(visible=False, width=1920, height=1080)
vis.add_geometry(pcd_vox)
vis.get_render_option().background_color = np.array([1, 1, 1])
vis.poll_events()
vis.update_renderer()
vis.capture_screen_image("classified_voxels.png", do_render=True)
vis.destroy_window()
print("✅ Saved voxel classification → classified_voxels.png")

# Point classification
voxel_tree = cKDTree(voxel_centers)
_, indices = voxel_tree.query(filtered_points_final, k=1)
point_labels = df["pred_label"].values[indices]
colors_points = np.array([color_map.get(lbl, [0.6, 0.6, 0.6]) for lbl in point_labels])

pcd_points = o3d.geometry.PointCloud()
pcd_points.points = o3d.utility.Vector3dVector(filtered_points_final)
pcd_points.colors = o3d.utility.Vector3dVector(colors_points)

vis = o3d.visualization.Visualizer()
vis.create_window(visible=False, width=1920, height=1080)
vis.add_geometry(pcd_points)
vis.get_render_option().background_color = np.array([1, 1, 1])
vis.poll_events()
vis.update_renderer()
vis.capture_screen_image("classified_points.png", do_render=True)
vis.destroy_window()
print("✅ Saved full point classification → classified_points.png")


=== STEP 8: Predict all voxels and points ===
pred_label
tree        12326
building    11071
terrain      3220
Name: count, dtype: int64
✅ Saved voxel classification → classified_voxels.png
✅ Saved full point classification → classified_points.png


In [18]:

# =====================================================
# 9️⃣ MANUAL VALIDATION (20 voxels per class)
# =====================================================
print("\n=== STEP 9: Manual evaluation (20 voxels per class) ===")

rng = np.random.default_rng(42)
for cls in ["building", "tree", "terrain"]:
    cls_idx = np.where(df["pred_label"] == cls)[0]
    if len(cls_idx) == 0:
        continue
    sample_idx = rng.choice(cls_idx, size=min(20, len(cls_idx)), replace=False)
    y_true = df.iloc[sample_idx]["label"]
    y_pred = df.iloc[sample_idx]["pred_label"]
    cm = confusion_matrix(y_true, y_pred, labels=["building", "tree", "terrain"])
    print(f"\nConfusion matrix for {cls} samples:")
    print(pd.DataFrame(cm, index=["building", "tree", "terrain"],
                       columns=["building", "tree", "terrain"]))
    print(classification_report(y_true, y_pred, zero_division=0))



=== STEP 9: Manual evaluation (20 voxels per class) ===

Confusion matrix for building samples:
          building  tree  terrain
building         2     0        0
tree             0     0        0
terrain          0     0        0
              precision    recall  f1-score   support

    building       0.10      1.00      0.18         2
   unlabeled       0.00      0.00      0.00        18

    accuracy                           0.10        20
   macro avg       0.05      0.50      0.09        20
weighted avg       0.01      0.10      0.02        20


Confusion matrix for tree samples:
          building  tree  terrain
building         0     0        0
tree             0     1        0
terrain          0     0        0
              precision    recall  f1-score   support

        tree       0.05      1.00      0.10         1
   unlabeled       0.00      0.00      0.00        19

    accuracy                           0.05        20
   macro avg       0.03      0.50      0.05       